# Text Classification
## Building  First Machine Learning Model to Classify Text
### Comparing TF-IDF vs Word Embeddings with Logistic Regression

##  Table of Contents

1. **What is Text Classification?** - Understanding the problem
2. **What is the 20 Newsgroups Dataset?** - Our data source
3. **The Two Approaches We'll Compare:**
   - TF-IDF (Term Frequency-Inverse Document Frequency)
   - Word Embeddings (GloVe vectors)
4. **Step-by-Step Implementation**
5. **Results & Comparison**
6. **Try It Yourself!**

---

## What is Text Classification?

**Text classification** is the task of automatically assigning a **category** (or "label") to a piece of text.

### Real-World Examples:

| Input Text | Task | Output |
|------------|------|--------|
| "I love this product!" | Sentiment Analysis | Positive ✅ |
| "Meeting at 3pm tomorrow" | Email Categorization | Calendar/Schedule 📅 |
| "How do I reset my password?" | Support Ticket Routing | Account Issues 🔑 |
| "Breaking: Stock market hits record high" | News Classification | Finance 📈 |

### In This Notebook:
We'll classify **newsgroup posts** into categories like:
- Sports (baseball, hockey)
- Science (medicine, space)

The computer will learn to read a post and decide: *"Is this about baseball, hockey, medicine, or space?"*






## What is the 20 Newsgroups Dataset?

The **20 Newsgroups dataset** is a classic dataset used for text classification research. It contains approximately **20,000 newsgroup posts** organized into **20 different topics**.

### What's a "Newsgroup"?

Before social media and Reddit, people used **newsgroups** (kind of like online forums) to discuss topics. Each newsgroup was dedicated to a specific subject:

- `rec.sport.baseball` → Baseball discussions
- `sci.space` → Space and astronomy discussions
- `comp.graphics` → Computer graphics discussions
- etc.

### Why Use This Dataset?

1. **It's built into scikit-learn** - Easy to load, no downloads needed!
2. **Real text data** - Messy, natural language (not artificially cleaned)
3. **Multi-class problem** - More interesting than just positive/negative
4. **Small enough to run quickly** - Great for learning!

### For This Tutorial:
We'll use **only 4 categories** to keep things simple and fast:
- 🏀 `rec.sport.baseball`
- 🏒 `rec.sport.hockey`  
- 🏥 `sci.med` (medicine)
- 🚀 `sci.space`


---

## The Machine Learning Pipeline (Big Picture)

Before we dive into code, let's understand the **overall process**:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        MACHINE LEARNING PIPELINE                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   1. RAW TEXT          2. PREPROCESSING        3. FEATURE EXTRACTION       │
│   ─────────────        ───────────────         ──────────────────────      │
│   "The quick brown     "quick brown fox"       [0.2, 0.0, 0.8, ...]        │
│    fox jumps..."       (cleaned tokens)        (numbers!)                  │
│                                                                             │
│                                                                             │
│   4. TRAIN MODEL       5. MAKE PREDICTIONS     6. EVALUATE                 │
│   ──────────────       ──────────────────      ────────────                │
│   Learn patterns       New text → Category     How accurate?               │
│   from examples                                                            │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### The Key Challenge: Computers Don't Understand Text!

Computers work with **numbers**, not words. So we need to convert text into numbers somehow. This is called **feature extraction** or **vectorization**.

**This notebook compares TWO different ways to convert text → numbers:**

1. **TF-IDF** (word frequency statistics)
2. **Word Embeddings** (learned semantic representations)

---

## The Two Approaches: TF-IDF vs Word Embeddings

### Approach 1: TF-IDF (Term Frequency-Inverse Document Frequency)

**What it does:** Converts each document into a vector where each dimension represents a word, and the value represents how "important" that word is to the document.

**Intuition:**
- Words that appear **frequently in a document** are probably important to that document
- BUT words that appear in **every document** (like "the", "is", "and") are not very informative
- TF-IDF balances these two ideas!

**Example:**
```
Document: "The NASA spacecraft landed on Mars"

TF-IDF might give:
  'nasa'      → 0.45  (important! specific to this doc)
  'spacecraft'→ 0.42  (important! specific)
  'mars'      → 0.38  (important! specific)
  'the'       → 0.01  (not important - appears everywhere)
  'on'        → 0.02  (not important - common word)
```

---

### Approach 2: Word Embeddings (GloVe)

**What it does:** Uses pre-trained vectors where each word is represented as a dense vector of ~100 numbers. Words with similar meanings have similar vectors.

**Intuition:**
- "king" and "queen" should have similar vectors (both royalty)
- "dog" and "puppy" should be close (both canines)
- "car" and "sandwich" should be far apart (unrelated)

**Example:**
```
'king'   → [0.2, -0.4, 0.8, 0.1, ...]  (100 numbers)
'queen'  → [0.3, -0.3, 0.7, 0.2, ...]  (similar to king!)
'apple'  → [-0.5, 0.2, 0.1, -0.8, ...] (very different)
```

For a **document**, we'll average all the word vectors together.

---

## Let's Start Coding!

Now that you understand the concepts, let's implement everything step by step.

**What we'll do:**
1. Install and import necessary libraries
2. Load the dataset and explore it
3. Preprocess the text (clean it up)
4. Build TF-IDF features and train a model
5. Build embedding features and train another model
6. Compare the results!

##  Setup: Installing Required Libraries

We need a few Python libraries:

| Library | Purpose |
|---------|--------|
| **scikit-learn** | Machine learning (already installed in Colab) |
| **spaCy** | Text preprocessing (tokenization, lemmatization) |
| **gensim** | Loading pre-trained word embeddings |
| **numpy** | Numerical operations (already installed) |

**Run the cell below to install what we need:**

---

## Imports: Loading Our Tools

Now we import all the libraries we'll use. Think of this like getting all your cooking ingredients ready before you start cooking!

**What each import does:**
- `numpy` - Math operations on arrays of numbers
- `sklearn` - Machine learning library (models, metrics, data)
- `gensim` - For loading word embeddings
- `spacy` - For text preprocessing


In [3]:
# Core python
import re
import numpy as np
from collections import Counter

# Data
from sklearn.datasets import fetch_20newsgroups

# Model+ evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

#TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer

# Embedding 
import gensim.downloader as api

# preprpcessing
import spacy

# Make results reproducible 
RANDOM_SEED =42
np.random.seed(RANDOM_SEED)



---

## Load and Explore Data

Now we load our dataset! This is always the first real step in any ML project.

### What we're loading:
We will load **only 4 categories** to keep things simple:

| Category | What it contains |
|----------|------------------|
| 🏀 `rec.sport.baseball` | Baseball discussions, scores, players |
| 🏒 `rec.sport.hockey` | Hockey discussions, NHL, players |
| 🏥 `sci.med` | Medical topics, health, diseases |
| 🚀 `sci.space` | Space exploration, NASA, astronomy |

**Why only 4?** Using fewer categories makes the notebook run faster and easier to understand. The same techniques work for all 20 categories!

``` text
20 Newsgroups Dataset
        ↓
Select 4 categories
        ↓
Remove headers / footers / quotes
        ↓
Load documents
        ↓
newsgroups
        │
        ├── .data ───────→ texts
        │                  actual documents
        │
        ├── .target ─────→ labels
        │                  0, 1, 2, 3
        │
        └── .target_names → category names
````

In [ ]:
# choose a small set og categories
# This defines a Python list of category names to work with. 
# The 20 Newsgroups dataset originally contains 20 different newsgroup topics.
# we are doing a 4-class problem instead of 20-class.
categories = [ "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.med",
    "sci.space",
]
# Load the dataset
from sklearn.datasets import fetch_20newsgroups 
#The function downloads/loads the 20 Newsgroups text dataset.
newsgroups = fetch_20newsgroups(subset='all',# dataset is usually us combination of both train anf test and we use both)
                                categories=categories,
                                remove=("headers", "footers", "quotes"),)# remove=... strips emails/headers/quotes to reduce "metadata noise"

texts = newsgroups.data                 # raw text documents (list of strings),contains the actual documents.
labels = newsgroups.target              # numeric labels (0..3),Every document needs a target/label so that a classification model knows its correct category.
target_names = newsgroups.target_names  # label -> category name

print("Number of documents:", len(texts)) # len(texts) counts how many documents are in the dataset.
print("Categories:", target_names) # This prints your four category names.
print("Label IDs:", sorted(set(labels)))



Number of documents: 3970
Categories: ['rec.sport.baseball', 'rec.sport.hockey', 'sci.med', 'sci.space']
Label IDs: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


==============================================================
### 2.1 Look at sample documents
Let's print a short snippet from one document in each category.

==============================================================

In [8]:
# Find one example index for each class label
example_indices = []
for class_id in range(len(target_names)):
    idx = np.where(labels == class_id)[0][0]
    example_indices.append(idx)

# Print snippets
for class_id, idx in enumerate(example_indices):
    print("=" * 80)
    print("Category:", target_names[class_id])
    print("Document index:", idx)
    print("-" * 80)
    print(texts[idx][:600])  # first 600 characters
    print()
### 2.2 Class distribution
# Before modeling, it's good to check whether classes are balanced.
# Count examples per class
counts = Counter(labels)

print("Class distribution:")
for class_id, count in sorted(counts.items()):
    print(f"  {class_id}: {target_names[class_id]:20s} -> {count} documents")


Category: rec.sport.baseball
Document index: 1
--------------------------------------------------------------------------------
Name            Pos   AB    H    2B    3B    HR    RBI    RS    SB    E    AVG
------------------------------------------------------------------------------
Boston          OF    12    7                        2     6              .583
Galarraga       1B    28   13     3           1      9     2              .464
Tatum           3B     5    2     1                                       .400
Cole            CF    24    9           1            2     8     2        .375
E. Young        2B    28    9     1     1     1      5    10     5    3   .321
Hayes           3B    25    7     1           2

Category: rec.sport.hockey
Document index: 6
--------------------------------------------------------------------------------
[more about the Messier-Samuelsson incident]
 I agree with Rick that Ulf's cross check wasn't illegal. It was the kind
 of check you see a dozen

===================================================

---

## Preprocessing: Cleaning Our Text

Before we can feed text to a machine learning model, we need to **clean it up**. This is called **preprocessing**.

### Why Preprocess?

Raw text is messy! Consider this example:

```
Original: "The Dogs were RUNNING quickly!!!"
```

**Problems:**
- "Dogs" and "dogs" are the same word but look different
- "RUNNING" and "running" are the same
- "!!!" punctuation doesn't help classification
- "running", "runs", "ran" all mean the same thing

**After preprocessing:**
```
Cleaned: "dog be run quickly"
```

### Our Preprocessing Steps:

| Step | What it does | Example |
|------|-------------|--------|
| 1. **Lowercasing** | Makes everything lowercase | "HELLO" → "hello" |
| 2. **Tokenization** | Splits text into words | "hello world" → ["hello", "world"] |
| 3. **Remove punctuation** | Removes !, ?, . etc. | "hello!" → "hello" |
| 4. **Lemmatization** | Converts words to base form | "running" → "run" |

### What is Lemmatization? 🤔

**Lemmatization** converts words to their "dictionary form" (called the *lemma*):

| Original Word | Lemma |
|---------------|-------|
| running, runs, ran | run |
| better, best | good |
| dogs, dog's | dog |
| was, were, is | be |

This helps because "running" and "runs" should be treated as the same word!

===========================================================================

### 3.1 Load spaCy Model

**spaCy** is a powerful library that knows how to:
- Split text into words (tokenization)
- Find the base form of words (lemmatization)
- Identify parts of speech (noun, verb, etc.)

We'll load a small English model (`en_core_web_sm`).

In [9]:
# Load spaCy English model
# We only need tokenization + lemmatization (no NER, no dependency parsing)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

print("spaCy pipeline components:", nlp.pipe_names)


spaCy pipeline components: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


=================================================
### 3.2 Define a preprocessing function
This function converts raw text into a list of **clean lemma tokens**.


In [11]:
def preprocess_text(text):
    '''
    Convert raw text -> list of lemmatized tokens.

    Steps:
    - lowercase
    - spaCy tokenize
    - keep only alphabetic tokens (removes numbers/punctuation)
    - lemmatize (base form of word)
    '''
    # 1) Lowercase
    text = text.lower()

    # 2) Run spaCy
    doc = nlp(text)

    tokens = []
    for token in doc:
        # Skip spaces and punctuation
        if token.is_space or token.is_punct:
            continue

        # Keep only "word-like" tokens (alphabetic)
        if not token.is_alpha:
            continue

        # Lemmatize (e.g., "running" -> "run")
        lemma = token.lemma_.strip()

        # spaCy sometimes uses "-PRON-" for pronouns in older models;
        # if that happens, fall back to the original token text.
        if lemma == "-PRON-" or lemma == "":
            lemma = token.text

        tokens.append(lemma)

    return tokens


def tokens_to_string(tokens):
    """Join tokens back into a single string (useful for TF-IDF)."""
    return " ".join(tokens)

=============================================

### 3.3 Try preprocessing on one document
We will compare the original text to the cleaned tokens.

In [12]:
sample_text = texts[0]

print("ORIGINAL (first 400 chars):")
print(sample_text[:400])
print()

sample_tokens = preprocess_text(sample_text)

print("PREPROCESSED TOKENS (first 40):")
print(sample_tokens[:40])
print()
print("Number of tokens:", len(sample_tokens))


ORIGINAL (first 400 chars):

Hum, do you enjoy putting words in my mouth? 
Come to Nome and meet some of these miners.. I am not sure how things go down
south in the lower 48 (I used to visit, but), of course to believe the
media/news its going to heck (or just plain crazy). 
Well it seems that alot of Unionist types seem to think that having a job is a
right, and not a priviledge. Right to the same job as your forbearers, S

PREPROCESSED TOKENS (first 40):
['hum', 'do', 'you', 'enjoy', 'put', 'word', 'in', 'my', 'mouth', 'come', 'to', 'nome', 'and', 'meet', 'some', 'of', 'these', 'miner', 'I', 'be', 'not', 'sure', 'how', 'thing', 'go', 'down', 'south', 'in', 'the', 'low', 'I', 'use', 'to', 'visit', 'but', 'of', 'course', 'to', 'believe', 'the']

Number of tokens: 151


=============================================
### 3.3 Try preprocessing on one document
We will compare the original text to the cleaned tokens.

In [13]:
sample_text = texts[0]

print("ORIGINAL (first 400 chars):")
print(sample_text[:400])
print()

sample_tokens = preprocess_text(sample_text)

print("PREPROCESSED TOKENS (first 40):")
print(sample_tokens[:40])
print()
print("Number of tokens:", len(sample_tokens))


ORIGINAL (first 400 chars):

Hum, do you enjoy putting words in my mouth? 
Come to Nome and meet some of these miners.. I am not sure how things go down
south in the lower 48 (I used to visit, but), of course to believe the
media/news its going to heck (or just plain crazy). 
Well it seems that alot of Unionist types seem to think that having a job is a
right, and not a priviledge. Right to the same job as your forbearers, S

PREPROCESSED TOKENS (first 40):
['hum', 'do', 'you', 'enjoy', 'put', 'word', 'in', 'my', 'mouth', 'come', 'to', 'nome', 'and', 'meet', 'some', 'of', 'these', 'miner', 'I', 'be', 'not', 'sure', 'how', 'thing', 'go', 'down', 'south', 'in', 'the', 'low', 'I', 'use', 'to', 'visit', 'but', 'of', 'course', 'to', 'believe', 'the']

Number of tokens: 151


=================================================
### 3.4 Preprocess the whole dataset
We will store:
- `all_tokens`: list of token lists (one list per document)
- `all_texts_clean`: list of strings (tokens joined back together)


In [14]:
# Preprocess every document
all_tokens = [preprocess_text(t) for t in texts]

# Join tokens into strings (handy for TF-IDF)
all_texts_clean = [tokens_to_string(toks) for toks in all_tokens]

print("Example cleaned text (first 200 chars):")
print(all_texts_clean[0][:200])
print()
print("Number of documents preprocessed:", len(all_texts_clean))

Example cleaned text (first 200 chars):
hum do you enjoy put word in my mouth come to nome and meet some of these miner I be not sure how thing go down south in the low I use to visit but of course to believe the medium news its go to heck 

Number of documents preprocessed: 3970


---

## Train/Test Split: Preparing Data for Learning

Before training a model, we need to split our data into two parts:

### Why Split the Data?

```
┌──────────────────────────────────────────────────────────────────┐
│                        ALL DATA (100%)                           │
├────────────────────────────────────┬─────────────────────────────┤
│     TRAINING SET (80%)             │     TEST SET (20%)          │
│     ───────────────────            │     ────────────────        │
│     Model learns from this         │     Model is evaluated      │
│     (sees the answers)             │     (never seen before!)    │
└────────────────────────────────────┴─────────────────────────────┘
```

**Think of it like studying for an exam:**
- **Training set** = Textbook & practice problems (you study from these)
- **Test set** = The actual exam (you've never seen these questions)

If you tested on the same questions you studied, you'd get 100% every time! But that doesn't mean you actually learned anything.

### What is "Stratification"?

When we split, we use `stratify=labels` to ensure each set has the **same proportion of categories**:
- If original data is 25% baseball, 25% hockey, 25% medical, 25% space
- Train set will also be ~25% each
- Test set will also be ~25% each

This prevents unlucky splits where, for example, all space articles end up in the test set.

In [15]:
# Split tokens and labels (stratify keeps class balance similar in train/test)
X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    all_tokens,
    labels,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=labels,
)

# For TF-IDF, we also want string versions
X_train_text = [tokens_to_string(toks) for toks in X_train_tokens]
X_test_text = [tokens_to_string(toks) for toks in X_test_tokens]

print("Train set size:", len(X_train_text))
print("Test set size :", len(X_test_text))


Train set size: 3176
Test set size : 794


---

## Approach 1: TF-IDF + Logistic Regression

Now we implement our **first approach** to text classification!

### What is TF-IDF?

**TF-IDF** stands for **Term Frequency - Inverse Document Frequency**. It's a way to convert text into numbers based on word statistics.

**The Formula (simplified):**
```
TF-IDF(word, document) = TF(word, document) × IDF(word)

Where:
- TF  = How many times the word appears in THIS document
- IDF = How rare the word is across ALL documents
```

### Why This Works:

| Word | TF (in doc) | IDF (overall) | TF-IDF | Meaning |
|------|-------------|---------------|--------|--------|
| "nasa" | High | High (rare) | **HIGH** | Important & specific! |
| "the" | High | Low (common) | **LOW** | Common word, not useful |
| "quasar" | Low | High (rare) | Medium | Rare, but not in this doc |

**Result:** Each document becomes a long vector of numbers (one per unique word in the vocabulary).

### 5.1 Build TF-IDF Vectors

We use scikit-learn's `TfidfVectorizer` which does all the math for us!

**Important:** We only `fit` on training data (to learn the vocabulary), then `transform` both train and test.

```
Training data → fit_transform() → Learn vocabulary + convert to numbers
Test data     → transform()     → Convert using SAME vocabulary
```

**Why?** We don't want to "cheat" by learning anything from the test set!


In [20]:
# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit on training text, transform both train and test
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("TF-IDF train matrix shape:", X_train_tfidf.shape)
print("TF-IDF test matrix shape :", X_test_tfidf.shape)

# Peek at a few learned feature names (words)
feature_names = tfidf_vectorizer.get_feature_names_out()
print("Number of TF-IDF features:", len(feature_names))
print("First 20 features:", feature_names[:20])


TF-IDF train matrix shape: (3176, 23009)
TF-IDF test matrix shape : (794, 23009)
Number of TF-IDF features: 23009
First 20 features: ['aa' 'aaa'
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg'
 'aaaaarrrrgh' 'aaaggghhh' 'aaai' 'aac' 'aagain' 'aan' 'aanerud'
 'aangegeven' 'aantal' 'aao' 'aargh' 'aarhu' 'aaron' 'aaronson'
 'aaroundpluto' 'aarseth' 'aas']


### 5.2 Train Logistic Regression (TF‑IDF features)

In [21]:
# Create and train the classifier
tfidf_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
tfidf_clf.fit(X_train_tfidf, y_train)

print("TF-IDF model trained!")


TF-IDF model trained!


---
### 5.3 Evaluate TF‑IDF model
We will report:
- **Accuracy** (overall correct predictions)
- **Classification report** (precision/recall/F1 per class)

In [22]:
# Predict on test set
tfidf_preds = tfidf_clf.predict(X_test_tfidf)

# Accuracy
tfidf_acc = accuracy_score(y_test, tfidf_preds)
print("TF-IDF + Logistic Regression accuracy:", tfidf_acc)
print()

# Detailed report
print(classification_report(y_test, tfidf_preds, target_names=target_names))

TF-IDF + Logistic Regression accuracy: 0.8879093198992444

                    precision    recall  f1-score   support

rec.sport.baseball       0.82      0.91      0.86       199
  rec.sport.hockey       0.96      0.85      0.90       200
           sci.med       0.90      0.92      0.91       198
         sci.space       0.89      0.87      0.88       197

          accuracy                           0.89       794
         macro avg       0.89      0.89      0.89       794
      weighted avg       0.89      0.89      0.89       794



---

## Approach 2: Word Embeddings + Logistic Regression

Now let's try a completely different approach!

### What are Word Embeddings?

**Word embeddings** represent words as **dense vectors of numbers** (typically 50-300 numbers per word). These vectors are learned by analyzing millions of sentences to understand which words appear in similar contexts.

### Key Insight: Similar Words Have Similar Vectors

```
'king'   → [0.2,  -0.4,  0.8,  0.1,  ...] (100 numbers)
'queen'  → [0.25, -0.35, 0.75, 0.15, ...] (similar to king!)
'apple'  → [-0.5,  0.2,  0.1,  -0.8, ...] (very different)
```

**This is powerful because:**
- The model knows "doctor" and "physician" mean similar things
- It can generalize to words it's never seen in training
- It captures semantic meaning, not just word counts

### How We'll Represent Documents:

For a whole document, we'll use a simple but effective method:

```
Document: "The doctor treated the patient"

1. Get vector for each word:
   'doctor'  → [0.1, 0.3, -0.2, ...]
   'treat'   → [0.2, 0.1,  0.0, ...]
   'patient' → [0.0, 0.4, -0.1, ...]

2. Average them:
   Document vector = AVERAGE of all word vectors
                   = [(0.1+0.2+0.0)/3, (0.3+0.1+0.4)/3, ...]
```

### What are GloVe Embeddings?

**GloVe** (Global Vectors for Word Representation) is a popular set of pre-trained word embeddings created by Stanford. They trained on 6 billion words from Wikipedia and news articles!

We'll use `glove-wiki-gigaword-100`:
- Trained on Wikipedia + news
- 100 dimensions per word
- ~400,000 words in vocabulary


### 6.1 Load Pre-trained Word Vectors

We use `gensim` to download GloVe embeddings. This might take a minute the first time!

In [23]:
# Download/load word vectors
# This returns a KeyedVectors object: you can look up vectors by word.
word_vectors = api.load("glove-wiki-gigaword-100")

print("Loaded word vectors!")
print("Vector size:", word_vectors.vector_size)
print("Example words in vocab:", list(word_vectors.key_to_index)[:10])

[==================================================] 100.0% 128.1/128.1MB downloaded
Loaded word vectors!
Vector size: 100
Example words in vocab: ['the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s"]


### 6.2 Convert tokens -> document vectors
We will:
- Look up each token in the embedding vocabulary
- Skip words that are **out-of-vocabulary (OOV)**
- Average the vectors we find
- If we find no vectors (all OOV), return a zero vector

In [24]:
def document_vector(tokens, word_vectors):
    '''
    Convert a list of tokens -> one document vector by averaging word embeddings.
    - OOV words are skipped
    - If no words are in the vocabulary, returns a zero vector
    '''
    vectors = []

    for tok in tokens:
        if tok in word_vectors:
            vectors.append(word_vectors[tok])

    if len(vectors) == 0:
        # No known words: return all zeros
        return np.zeros(word_vectors.vector_size, dtype=np.float32)

    # Average across all word vectors we found
    return np.mean(vectors, axis=0)


# Build train/test matrices
X_train_emb = np.vstack([document_vector(toks, word_vectors) for toks in X_train_tokens])
X_test_emb = np.vstack([document_vector(toks, word_vectors) for toks in X_test_tokens])

print("Embedding train matrix shape:", X_train_emb.shape)
print("Embedding test matrix shape :", X_test_emb.shape)

Embedding train matrix shape: (3176, 100)
Embedding test matrix shape : (794, 100)


### 6.3 Train Logistic Regression (embedding features)

In [25]:
emb_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
emb_clf.fit(X_train_emb, y_train)

print("Embedding model trained!")

Embedding model trained!


### 6.4 Evaluate embedding model

In [26]:
emb_preds = emb_clf.predict(X_test_emb)

emb_acc = accuracy_score(y_test, emb_preds)
print("Embeddings + Logistic Regression accuracy:", emb_acc)
print()

print(classification_report(y_test, emb_preds, target_names=target_names))

Embeddings + Logistic Regression accuracy: 0.8425692695214105

                    precision    recall  f1-score   support

rec.sport.baseball       0.80      0.85      0.82       199
  rec.sport.hockey       0.87      0.81      0.84       200
           sci.med       0.88      0.85      0.87       198
         sci.space       0.83      0.85      0.84       197

          accuracy                           0.84       794
         macro avg       0.84      0.84      0.84       794
      weighted avg       0.84      0.84      0.84       794



--

## Comparison and Takeaways

**Congratulations!** You've built TWO text classification models! Now let's compare them.

### Side-by-Side Comparison:

| Aspect | TF-IDF | Word Embeddings |
|--------|--------|----------------|
| **Vector Size** | ~20,000+ (one per word) | 100 (fixed) |
| **What it captures** | Word frequency/importance | Semantic meaning |
| **Out-of-vocabulary words** | Ignored | Ignored |
| **Speed** | Very fast | Fast (after loading) |
| **Interpretability** | High (you can see which words matter) | Low (dense vectors) |

### Questions to Ponder:
- Which one performed better on this dataset?
- Why might TF‑IDF win on topic classification?
- When might embeddings be a better choice?


In [27]:
print("Final comparison (same train/test split):")
print(f"  TF-IDF + Logistic Regression accuracy      : {tfidf_acc:.4f}")
print(f"  Embeddings (avg) + Logistic Regression acc : {emb_acc:.4f}")

if tfidf_acc > emb_acc:
    print("\nTF-IDF performed better on this run.")
elif emb_acc > tfidf_acc:
    print("\nEmbeddings performed better on this run.")
else:
    print("\nThey performed the same on this run.")

Final comparison (same train/test split):
  TF-IDF + Logistic Regression accuracy      : 0.8879
  Embeddings (avg) + Logistic Regression acc : 0.8426

TF-IDF performed better on this run.


### Why results often look like this
- **TF‑IDF** is very strong for **topic classification** because specific words (like *orbit*, *nasa*, *doctor*, *symptom*) are highly informative for these categories.  
- **Averaging embeddings** can lose information about which specific words appear, because it blends them into one vector.  
- Embeddings can shine when you care more about **semantic similarity** (meaning), or when you have less labeled data and want general-purpose representations.

**When to choose each approach**
- Choose **TF‑IDF** when:
  - You want a fast, strong baseline
  - Your task is mostly about **keywords/topics**
  - You want simple interpretability (you can inspect important words)
- Choose **embeddings** when:
  - You want models to generalize across similar words (e.g., *physician* vs *doctor*)
  - You care about meaning beyond raw word counts
  - You plan to build more advanced models later (embeddings are a common starting point)


``` text
Raw text + labels
      ↓
Preprocessing
      ↓
Clean tokens
      ↓
Train / test split
      ↓
Convert text → numbers
      ↓
 ┌─────────────┬──────────────┐
 │   TF-IDF    │    GloVe     │
 │             │  Embeddings  │
 └──────┬──────┴──────┬───────┘
        ↓             ↓
 Logistic Regression  Logistic Regression
        ↓             ↓
    Predictions    Predictions
        ↓             ↓
       Accuracy / Classification Report
        ↓
        Compare
```